# 07 · Channel coding

A forward error correction (FEC) code adds redundancy so that the receiver
can remove the errors the channel left.  This notebook covers:

1. codes as objects, `encode` and `decode`, pre- versus post-FEC metrics, and
   block sizes: how codewords fill symbols;
2. hard and soft decoding of a block code (Hamming, Chase), error-rate
   curves and the decoding outcome;
3. algebraic codes: Reed-Solomon and the bounded-distance threshold;
4. LDPC codes and iterative decoding: the waterfall;
5. **NGMI**, the metric that predicts whether a soft-decision code decodes,
   and the EXIT chart that predicts its threshold from the code;
6. polar, convolutional and turbo codes in one comparison against Eb/N0,
   the effect of block length, and an EXIT chart of iterative demapping.

It builds on the LLRs and the GMI of
[06 · AWGN and performance metrics](06-awgn-and-performance-metrics.ipynb).

In [ ]:
from math import comb

import matplotlib.pyplot as plt
import numpy as np

import commkit as ck
from commkit import Constellation
from commkit.backend import to_device
from commkit.coding import (
    BCJR,
    SCL,
    Chase,
    ConvolutionalCode,
    LDPCCode,
    LinearBlockCode,
    MaximumLikelihood,
    MinSum,
    PolarCode,
    ReedSolomonCode,
    SumProduct,
    Syndrome,
    TurboCode,
    TurboIterations,
    decode,
    decoder_exit_curve,
    demapper_exit_curve,
    encode,
    exit_chart,
    ldpc_exit_curves,
    ldpc_threshold,
)

DEVICE = "gpu" if ck.backend.is_cupy_available() else "cpu"
print(DEVICE)

ck.plotting.apply_default_theme()
QPSK = Constellation.qam(4)


def host(a):
    return to_device(a, "cpu")


def coded_link(code, esn0_db, num_codewords, constellation=QPSK, seed=0):
    """A coded payload over AWGN at one sample per symbol, and its LLRs."""
    tx = ck.generate(
        constellation,
        symbol_rate=1e9,
        code=code,
        num_blocks=num_codewords,
        rng=seed,
        device=DEVICE,
    )
    rx = ck.impairments.apply_awgn(tx, esn0_db=esn0_db, rng=seed + 1)
    return rx, ck.mapping.compute_llr(rx, noise_var=10 ** (-esn0_db / 10))

## 1. Codes as objects

A code is a frozen value object with `n` transmitted and `k` information
bits per block.  `encode` maps information bits to codewords and `decode`
maps LLRs (or hard bits) back, returning a `DecodeResult` with the
decoded bits, the decoder's own success check (`valid`) and the iterations
used.  The decoder is chosen by a method object, as everywhere in CommKit.

In [ ]:
h74 = LinearBlockCode.hamming(3)
print(h74, "rate", h74.rate)
u = np.array([1, 0, 1, 1], dtype=np.int8)
c = encode(u, code=h74)
print("codeword:", c)
for pos in range(7):  # every single error is corrected
    e = c.copy()
    e[pos] ^= 1
    assert np.array_equal(decode(e, code=h74).bits, u)
print("all 7 single errors corrected")

`generate(..., code=)` builds a coded payload.  Its reference keeps two
bit streams: `bits`, the labels on the air, and `info_bits`, the source
bits.  Pre-FEC metrics compare the hard decisions with `bits`; post-FEC
metrics compare the decoder's output with `info_bits`.  For an uncoded
payload `info_bits` *is* `bits`, so the same calls work unchanged.

A decoder works on whole codewords, so the record is sized in them:
`num_blocks=` is the number of codewords per channel, and `generate`
computes the symbols they fill.  To fit a fixed record length instead (an
arbitrary waveform generator's memory, say), pass `num_symbols` with
`pad=True`: the record carries as many whole codewords as fit, then random
filler bits; `reference.num_blocks` counts the codewords, and
`decode(llrs[..., : ref.num_blocks * code.n], code=code)` decodes them.

In [ ]:
rx, llrs = coded_link(h74, esn0_db=9.0, num_codewords=20000)
ref = rx.reference
print("code block (n, k):", ref.code_block)
res = decode(llrs, code=h74)
print(f"pre-FEC BER  {ck.metrics.ber(rx):.2e}")
print(f"post-FEC BER {ck.metrics.ber(res.bits, ref.info_bits):.2e}")
print(f"post-FEC FER {ck.metrics.fer(res.bits, ref.info_bits, info_bits=4):.2e}")

### Block sizes: codewords, bits and symbols

Every code works on fixed blocks: `k` information bits in, `n` coded bits
out, rate `R = k / n`.  The block, a *codeword*, is fixed by the code (for
a standard code, by the standard), not by the record: a DVB-S2 normal
codeword has `n = 64800` bits and a short one 16200, at any rate; the rate
only decides how many of them are information.  A record carries a whole
number of codewords, because a decoder needs all `n` bits of one.

On the air, the `n` bits of a codeword are mapped `m = bits_per_symbol` at
a time, so a codeword fills `n / m` symbols: 16200 bits are 4050 16-QAM
symbols.  When `m` does not divide `n`, a codeword ends inside a symbol,
and the shortest record of whole codewords and whole symbols is
`lcm(n, m)` bits: Hamming(7, 4) with 16-QAM needs 4 codewords, 28 bits, 7
symbols.

One trap: the DVB-S2 *labels* of the short codes are not their rates.  The
short codes keep the labels of the normal ones, but `"1/2"` short has
`k = 7200` (rate 4/9, not 8100 bits) and `"5/6"` short `k = 13320` (rate
0.822, not 13500).  Read `code.k` and `code.rate`, never the label:

In [ ]:
from math import lcm

codes = {
    "Hamming(7, 4)": LinearBlockCode.hamming(3),
    "KP4 RS(544, 514)": ReedSolomonCode.kp4(),
    "polar NR (256, 128)": PolarCode.nr(k=128, n=256),
    'DVB-S2 "1/2" normal': LDPCCode.dvbs2("1/2"),
    'DVB-S2 "1/2" short': LDPCCode.dvbs2("1/2", "short"),
    'DVB-S2 "5/6" short': LDPCCode.dvbs2("5/6", "short"),
}
print(f"{'':42s}shortest record: codewords -> symbols")
header = f"{'n':>6s} {'k':>6s} {'rate':>6s}  {'QPSK':14s}{'16-QAM':14s}64-QAM"
print(f"{'':21s} {header}")
for name, code in codes.items():
    row = f"{name:21s} {code.n:6d} {code.k:6d} {code.rate:6.3f}  "
    for m in (2, 4, 6):
        bits = lcm(code.n, m)  # whole codewords in whole symbols
        row += f"{bits // code.n:>4d} -> {bits // m:<6d}"
    print(row)

`generate(..., num_blocks=)` and `SingleCarrierFrame(payload_blocks=)` do
this arithmetic: they take the number of codewords and compute the
symbols.  Eight short `"5/6"` codewords on 16-QAM, the payload of
[24](24-coherent-transceiver-end-to-end.ipynb), are 8 x 16200 / 4 = 32400
symbols.  A symbol count that does not hold whole codewords raises, naming
the nearest valid ones (in a frame, the payload must also fill whole pilot
periods):

In [ ]:
C16 = Constellation.qam(16)
code56 = LDPCCode.dvbs2("5/6", "short")
sig = ck.generate(C16, symbol_rate=1e9, code=code56, num_blocks=8)
ref = sig.reference
print(f"{ref.num_blocks} codewords x {code56.n} bits / 4 bits per symbol")
num_info = ref.info_bits.shape[-1]
print(f"= {sig.samples.shape[-1]} symbols, carrying {num_info} information bits")
try:
    ck.generate(C16, num_symbols=1000, symbol_rate=1e9, code=h74)
except ValueError as err:
    print(err)

## 2. Hard and soft decoding

Syndrome decoding uses only the hard decisions.  A soft decoder uses the
LLR magnitudes too.  Maximum-likelihood decoding tries all `2^k` codewords,
which is affordable only for small codes.  **Chase** decoding flips the
least reliable bits in all combinations and hard-decodes each test word.
It approaches ML at a fraction of the cost, and it is how the
Hamming(128,119) inner code of 400ZR is decoded in
[23 · FEC in optical standards](23-fec-in-optical-standards.ipynb).

`error_rate_curve` runs the experiment at every Es/N0 of a grid: random
information bits, encoding, mapping, AWGN, LLRs and decoding, in batches,
until 100 blocks are in error or `max_blocks` are decoded.  It returns the
counts as an `ErrorRateCurve`, which `plot_error_rate` draws: the post-FEC
bit error rate (solid), the frame error rate (dashed, with 95 % intervals)
and the pre-FEC bit error rate of the channel (dotted).

In [ ]:
h16 = LinearBlockCode.hamming(4, extended=True)  # (16, 11), distance 4
methods = {
    "syndrome (hard)": Syndrome(max_weight=1),
    "Chase, 3 positions": Chase(3, hard=Syndrome(max_weight=1), soft=False),
    "maximum likelihood": MaximumLikelihood(soft=False),
}
grid = np.arange(4.0, 10.1, 1.0)
curves = [
    ck.coding.error_rate_curve(h16, constellation=QPSK, esn0_db=grid, method=m,
                                max_blocks=20_000, label=name, rng=1, device=DEVICE)
    for name, m in methods.items()
]  # fmt: skip
ck.plotting.plot_error_rate(curves, show=True)
print("post-FEC BER at 7 dB:", {c.label: f"{c.ber[3]:.1e}" for c in curves})
syndrome = curves[0]
print(
    f"syndrome decoding at 6 dB: {syndrome.detected_failures[2]} blocks flagged, "
    f"{syndrome.undetected_errors[2]} wrong but reported valid, "
    f"of {syndrome.num_blocks[2]}"
)

At 7 dB soft decoding lowers the post-FEC BER more than tenfold.  Chase with
three test positions matches maximum-likelihood decoding on this code at a
fraction of the cost.

The right panel splits the failed blocks.  With `max_weight=1` the syndrome
decoder corrects single errors and *detects* double errors, which the
distance 4 of the extended code exposes: it flags most of its failures
(dashed), and a receiver can discard those blocks or ask for them again.
Maximum-likelihood decoding has no check at all, so each of its errors goes
undetected.  Chase has a check, but one of its test words always flips a
single bit, and that one always decodes, so it never flags a block either.
`DecodeResult.checked` says whether a decoder can flag failures.

Chase and maximum-likelihood decoding can also return soft output, the
a-posteriori LLRs an outer code or an iterative receiver needs (section
4 shows them for LDPC codes).  An error-rate curve needs only the
decisions, so the sweep passes `soft=False`: the same bits, in less than
half the time.

## 3. Algebraic codes: Reed-Solomon

A Reed-Solomon code over GF(2^m) corrects any `t = (n - k) / 2` wrong
*symbols* per codeword, whatever the bit errors inside them, and nothing
beyond: decoding is bounded-distance.  So the frame error rate is the
probability of more than `t` symbol errors, a binomial tail.  The KP4 code
RS(544, 514) of 200G-800G Ethernet has `t = 15` 10-bit symbols.  Its
threshold, a pre-FEC BER of 2.4e-4, leaves a frame error rate around 1e-12,
which is too rare to simulate but follows from the tail.

In [ ]:
kp4 = ReedSolomonCode.kp4()
print(kp4, "t =", kp4.t)


def kp4_fer(p_bit):
    p_sym = 1 - (1 - p_bit) ** 10
    return sum(
        comb(544, i) * p_sym**i * (1 - p_sym) ** (544 - i) for i in range(16, 545)
    )


rng = np.random.default_rng(0)
u = rng.integers(0, 2, (400, kp4.k), dtype=np.int8)
words = host(encode(u, code=kp4))
for p in (3e-3, 4e-3, 5e-3):
    flipped = words ^ (rng.random(words.shape) < p)
    res = decode(ck.backend.to_device(flipped, DEVICE), code=kp4)
    fer = float(np.mean(np.any(host(res.bits) != u, axis=1)))
    print(
        f"pre-FEC BER {p:.0e}: simulated FER {fer:.3f}, binomial tail {kp4_fer(p):.3f}"
    )
print(f"at the 2.4e-4 threshold the tail is {kp4_fer(2.4e-4):.1e}")

## 4. LDPC codes and iterative decoding

LDPC codes are decoded by passing messages along a sparse graph of parity
checks.  They decode almost error-free above a sharp threshold (the
*waterfall*), close to the GMI.  `LDPCCode.dvbs2` gives the DVB-S2 codes,
the common stand-in for the proprietary soft-decision FEC of coherent
modems.  `MinSum` (normalized min-sum) is the hardware decoder and
`SumProduct` exact belief propagation.  Both run layered, on the GPU as
one CUDA kernel.

In [ ]:
ldpc = LDPCCode.dvbs2("1/2", "short")
print(ldpc, f"rate {ldpc.rate:.3f}")
grid = np.arange(-0.5, 1.01, 0.25)
curves = [
    ck.coding.error_rate_curve(ldpc, constellation=QPSK, esn0_db=grid, method=m,
                                min_block_errors=20, max_blocks=60, label=name,
                                rng=2, device=DEVICE)
    for name, m in (("min-sum", MinSum()), ("sum-product", SumProduct()))
]  # fmt: skip
ck.plotting.plot_error_rate(curves, quantity="error_rate", show=True)
for i, e in enumerate(grid):
    print(
        f"{e:4.2f} dB: FER min-sum {curves[0].fer[i]:.3f}, sum-product "
        f"{curves[1].fer[i]:.3f} ({curves[1].mean_iterations[i]:.0f} iterations)"
    )

Points without a single block error are drawn as the upper end of their
95 % interval (triangles): 60 error-free frames still allow a frame error
rate up to 6 %.  To claim lower rates, decode more frames (`max_blocks`).

How noisy is the channel the code cleans up?  At the last point of the
sweep (Es/N0 = 1 dB) the QPSK clouds overlap into one blur and 13 % of the
bits arrive wrong.  The decoder returns not only bits but, for
belief propagation, the a-posteriori LLRs of all `n` transmitted bits
(`res.llrs`); `plot_llr_distribution` takes the `DecodeResult` with the
transmitted Signal.  The channel LLRs straddle zero, the decoded ones are
pushed far to the side of the correct bit value, because every parity
check now agrees.

In [ ]:
rx, llrs = coded_link(ldpc, grid[-1], num_codewords=24, seed=110)
res = decode(llrs, code=ldpc, method=SumProduct())
print(f"Es/N0 {grid[-1]:.2f} dB: pre-FEC BER {ck.metrics.ber(rx):.3f}, ", end="")
print(f"post-FEC BER {ck.metrics.ber(res, rx):.1e}")
fig, axes = plt.subplots(
    1, 2, figsize=np.multiply(plt.rcParams["figure.figsize"], (2, 1))
)
ck.plotting.plot_constellation(rx, ax=axes[0], title=f"QPSK at {grid[-1]:.0f} dB")
ck.plotting.plot_llr_distribution(
    llrs, rx.reference.bits, quantity="density", ax=axes[1], label="channel"
)
ck.plotting.plot_llr_distribution(
    res,
    rx,
    quantity="density",
    ax=axes[1],
    label="after sum-product",
    xlim=(-60, 60),
    title="LLRs given the transmitted bits",
)
plt.show()

## 5. NGMI: will the code decode?

The normalized GMI, `NGMI = 1 - (H(X) - GMI) / m`, is 1 for perfect LLRs.
A soft-decision code of rate `R` decodes once the NGMI exceeds a threshold
somewhat above `R`, and that threshold hardly depends on the modulation.
This is why optical systems quote FEC thresholds as NGMI.  For this
rate-1/2 code:

In [ ]:
for i, e in enumerate(grid):
    rx, _ = coded_link(ldpc, e, num_codewords=8, seed=120)
    ngmi = ck.metrics.ngmi(rx, noise_var=10 ** (-e / 10))
    print(f"Es/N0 {e:4.2f} dB: NGMI {ngmi:.3f}  FER {curves[1].fer[i]:.3f}")

The frames start decoding once the NGMI passes about 0.5, roughly 0.06
above the code rate (0.444).  The gap measures how far this code is from
capacity at its length of 16200 bits.  Sum-product decodes slightly
earlier than min-sum, about 0.1 dB, and costs more per iteration.

### Predicting the threshold: EXIT charts

The threshold follows from the code alone.  A parity-check matrix is a graph
of variable nodes (the codeword bits, columns) and check nodes (rows);
`plot_parity_check` shows DVB-S2's: an irregular information part and the
dual-diagonal (accumulator) parity part, with most bits in two or three
checks.

In [ ]:
ck.plotting.plot_parity_check(ldpc, show=True)

Belief propagation passes information back and forth between the two kinds
of nodes.  `ldpc_exit_curves` gives the transfer curve of each in the
Gaussian approximation: the extrinsic information of the variable nodes
rises with the information they receive and with the channel's
information per bit, here the NGMI of the uniform QPSK link.
`exit_chart` follows the exchange: decoding succeeds when the staircase
between the two curves gets through to the top right.  `ldpc_threshold`,
the channel information at which this tunnel opens, is predicted without
simulating a single frame, and it matches the NGMI at which the frames
above start to decode.

In [ ]:
variable, check = ldpc_exit_curves(ldpc, channel_mi=0.52)
chart = exit_chart(variable, check)
print(f"threshold: {ldpc_threshold(ldpc):.3f} bit (code rate {ldpc.rate:.3f})")
print(f"at an NGMI of 0.52 the tunnel is open: {chart.converges}")
ck.plotting.plot_exit_chart(chart, show=True)

## 6. Polar, convolutional and turbo codes

Three more families, each with its standard decoder: 5G NR polar codes
with CRC-aided list decoding, the classic `K = 7` convolutional code with
Viterbi decoding, and the LTE turbo code with iterative BCJR decoding.
All have rate about 1/2 or 1/3 and 128 information bits.  Codes of
different rates are compared at the same energy per *information* bit,
`Eb/N0 = Es/N0 - 10 log10(R m)` for `m` bits per symbol, which
`plot_error_rate(..., x_axis="ebn0")` puts on the axis:

In [ ]:
codes = {
    "polar (256, 128) + CRC-11, SCL(8)": (PolarCode.nr(k=128, n=256, crc="11"), SCL(8)),
    "convolutional K=7, Viterbi": (ConvolutionalCode.k7(128), None),
    "turbo LTE K=128, 8 iterations": (TurboCode.lte(128), TurboIterations()),
}
ebn0 = np.arange(0.5, 4.01, 0.5)


def sweep(code, method, label):
    esn0 = ebn0 + 10 * np.log10(2 * code.rate)  # QPSK: 2 bits per symbol
    return ck.coding.error_rate_curve(code, constellation=QPSK, esn0_db=esn0,
                                       method=method, min_block_errors=50,
                                       max_blocks=2000, label=label, rng=3,
                                       device=DEVICE)  # fmt: skip


curves = [sweep(code, method, name) for name, (code, method) in codes.items()]
ck.plotting.plot_error_rate(curves, x_axis="ebn0", pre_fec=False, show=True)
for c in curves:
    print(f"{c.label:34s} rate {c.rate:.3f}  FER at Eb/N0 2.5 dB: {c.fer[4]:.4f}")
bcjr = sweep(ConvolutionalCode.k7(128), BCJR(), "BCJR")
same = np.array_equal(bcjr.block_errors, curves[1].block_errors)
print(f"BCJR fails on as many blocks as Viterbi at every point: {same}")
polar = curves[0]
print(
    f"polar at Eb/N0 1.5 dB: {polar.detected_failures[2]} failures flagged by the "
    f"CRC, {polar.undetected_errors[2]} undetected, of {polar.num_blocks[2]} blocks"
)

At the same energy per information bit, the turbo code (rate 1/3) and the
polar list decoder lose the fewest frames, and the 64-state convolutional
code the most.  BCJR (not drawn) makes the same decisions as Viterbi here;
it also returns the soft output that iterative receivers need.  The CRC of
the polar code is its success check: the outcome panel shows that it flags
nearly every failed block, while the convolutional and turbo decoders have
no check and report every error as valid.

### Block length

Within a family, longer blocks decode closer to capacity: a decoder that
sees more bits averages the noise over more of them.  The LTE turbo code
at three of its block sizes, all at rate 1/3 (the capacity of a rate-1/3
binary code over AWGN is reached at Eb/N0 = -0.5 dB):

In [ ]:
ebn0 = np.arange(0.0, 2.01, 0.25)
lengths = []
for k in (128, 1024, 6144):
    code = TurboCode.lte(k)
    esn0 = ebn0 + 10 * np.log10(2 * code.rate)
    lengths.append(
        ck.coding.error_rate_curve(code, constellation=QPSK, esn0_db=esn0,
                                   min_block_errors=20, max_blocks=100,
                                   label=f"turbo K = {k}", rng=5, device=DEVICE)
    )  # fmt: skip
ck.plotting.plot_error_rate(
    lengths, x_axis="ebn0", quantity="error_rate", pre_fec=False, show=True
)
for c in lengths:
    print(
        f"{c.label:16s} FER at Eb/N0 0.5 dB: {c.fer[2]:.2f}, at 1.0 dB: {c.fer[4]:.2f}"
    )

The waterfall moves left and steepens with the block length: at Eb/N0 =
0.5 dB the frame error rate falls from 0.36 (K = 128) to 0.20 (K = 1024)
and 0.01 (K = 6144), and from 1 dB on the two longer codes decode every
frame simulated.  The short code keeps a slowly falling tail instead.
Longer blocks cost latency and decoder memory, which is why 5G uses polar
codes for its short control messages and LDPC codes for data.

### Iterative demapping, charted

EXIT charts are not limited to LDPC codes.  Any decoder that returns soft
output has a transfer curve, measured by feeding it a-priori LLRs of known
information (`decoder_exit_curve`).  So does the demapper, when the decoder
feeds its extrinsic LLRs back as a-priori information on the label's other
bits (`demapper_exit_curve`): iterative demapping, BICM-ID.  The rate-1/2
`K = 7` code with exact BCJR, and 16-QAM at Es/N0 = 6 dB with two
labellings:

In [ ]:
decoder = decoder_exit_curve(
    ConvolutionalCode.k7(1000), method=BCJR(method="exact"), rng=1, device=DEVICE
)
pts = Constellation.qam(16).points
natural = np.zeros((16, 4), dtype=np.int8)  # natural binary over the grid
natural[np.lexsort((pts.imag, pts.real))] = (
    np.arange(16)[:, None] >> np.arange(3, -1, -1)
) & 1
labellings = {
    "Gray": Constellation.qam(16),
    "natural": Constellation(pts, bit_labels=natural),
}
fig, axes = plt.subplots(
    1, 2, figsize=np.multiply(plt.rcParams["figure.figsize"], (2, 1))
)
for ax, (name, c) in zip(axes, labellings.items(), strict=True):
    demapper = demapper_exit_curve(c, noise_var=10 ** (-0.6), rng=2, device=DEVICE)
    chart = exit_chart(demapper, decoder)
    ck.plotting.plot_exit_chart(chart, ax=ax, title=f"{name} labels")
    print(
        f"{name:8s}: demapper {demapper.extrinsic[0]:.3f} without feedback, "
        f"{demapper.extrinsic[-1]:.3f} with the other bits known; the iteration "
        f"stalls with the decoder's output at {chart.trajectory[-1, 0]:.3f}"
    )
plt.show()

Gray labels give the most information without feedback but gain almost
nothing from it: iterating with the decoder is pointless, which is why
Gray-labelled BICM decodes once.  Natural labels start lower and gain
more, and the iteration ends with the decoder's output at 0.92 instead of
0.71.  Neither tunnel opens at this SNR: the stall is where the curves
cross.  Measured curves carry Monte Carlo error and assume Gaussian
a-priori LLRs and long interleavers; `commkit.coding.exit` lists the
limitations.

**Summary**

* A code is an object; `encode` and `decode` work on bits and LLRs, and
  `decode` takes a method object for the decoder.
* `generate(..., code=)` keeps both the bits on the air and the source
  bits, so pre-FEC (`ber(rx)`) and post-FEC (`ber(res, rx)`, `fer`) metrics
  are one call each.
* `coding.error_rate_curve` measures BER, FER, pre-FEC BER and the
  decoding outcome over a grid of Es/N0; `plot_error_rate` draws them,
  against Eb/N0 to compare codes of different rates.  `DecodeResult.checked`
  says whether a decoder can flag its failures.
* Hard decoding throws away the reliabilities; soft decoding (Chase, ML,
  belief propagation) uses them.  `soft=False` skips a decoder's soft
  output when only the decisions matter.
* Longer blocks of the same family decode closer to capacity.
* Bounded-distance RS codes have a binomial-tail threshold; LDPC codes a
  waterfall predicted by the NGMI, and by an EXIT chart from the code alone.
* EXIT charts of measured curves (`decoder_exit_curve`,
  `demapper_exit_curve`) show what iterating between two components gains.

**Next:** [08 · Probabilistic shaping and coded modulation](08-probabilistic-shaping-and-coded-modulation.ipynb).